# 🎬 AI Clipping Bot - Whop Campaign Automation
**Powered by Google Gemini 2.5 Flash + FFmpeg | 100% Gratuit**

---
### Comment utiliser ce notebook :
1. Remplissez vos clés dans la **Cellule 1 (Config)**
2. Cliquez sur **Runtime → Run All** (ou Ctrl+F9)
3. Vos clips seront dans votre **Google Drive** !

> ⚠️ Gardez l'onglet Colab ouvert pendant le traitement.
> 
> 💡 **Limites gratuites :** ~1500 requêtes/jour, ~10 requêtes/minute.
> Le bot gère automatiquement les pauses pour ne jamais dépasser ces limites.

In [ ]:
# ============================================================
# CELLULE 1 - CONFIGURATION (modifiez uniquement ici)
# ============================================================

GEMINI_API_KEY = 'COLLEZ_VOTRE_CLE_GEMINI_ICI'
GITHUB_TOKEN    = 'COLLEZ_VOTRE_TOKEN_GITHUB_ICI'
GITHUB_USER     = 'mrdarkness5298-blip'
GITHUB_REPO     = 'ai-clipping-bot'

BRIEF_URL = 'https://docs.google.com/document/d/1modTZq5jJ5WkYZ1TmitNcV7MfHxTTRyGfKsFLdqCqUM/edit'

CLIPS_TO_GENERATE = 20
MIN_CLIP_DURATION = 10
MAX_CLIP_DURATION = 55
DRIVE_OUTPUT_FOLDER = 'AI_Clips_Output'

# Validation des cles
assert GEMINI_API_KEY != 'COLLEZ_VOTRE_CLE_GEMINI_ICI', '\u274c Collez votre cle Gemini API dans GEMINI_API_KEY !'
assert GITHUB_TOKEN != 'COLLEZ_VOTRE_TOKEN_GITHUB_ICI', '\u274c Collez votre token GitHub dans GITHUB_TOKEN !'
print('\u2705 Configuration OK')

In [ ]:
# ============================================================
# CELLULE 2 - INSTALLATION
# ============================================================
print('Installation des dependances...')
!apt-get install -y ffmpeg -qq 2>/dev/null
!pip install -q --upgrade google-genai google-api-python-client 2>&1 | grep -v 'already satisfied'

import subprocess
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print(f'FFmpeg: {result.stdout.split(chr(10))[0]}')

from google import genai
print(f'google-genai SDK charge')
print('\u2705 Installation complete !')

In [ ]:
# ============================================================
# CELLULE 3 - CONNEXION GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

OUTPUT_DIR   = Path(f'/content/drive/MyDrive/{DRIVE_OUTPUT_FOLDER}')
DOWNLOAD_DIR = Path('/content/downloads')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(exist_ok=True)

print(f'\u2705 Drive monte ! Clips -> {OUTPUT_DIR}')

In [ ]:
# ============================================================
# CELLULE 4 - LECTURE DU BRIEF GOOGLE DOCS
# ============================================================
import re, requests
from html.parser import HTMLParser

def read_brief(doc_url):
    doc_id = re.search(r'/document/d/([a-zA-Z0-9_-]+)', doc_url).group(1)
    html = requests.get(
        f'https://docs.google.com/document/d/{doc_id}/export?format=html',
        timeout=30
    ).text

    class Parser(HTMLParser):
        def __init__(self):
            super().__init__()
            self.text, self.links = [], []
        def handle_starttag(self, tag, attrs):
            if tag == 'a':
                href = dict(attrs).get('href', '')
                if 'google.com/url?q=' in href:
                    from urllib.parse import unquote
                    m = re.search(r'q=([^&]+)', href)
                    if m: href = unquote(m.group(1))
                if href: self.links.append(href)
        def handle_data(self, d):
            self.text.append(d)

    p = Parser()
    p.feed(html)
    text = ' '.join(p.text)
    drive_links = []
    for href in p.links:
        if 'drive.google.com' in href and href not in drive_links:
            drive_links.append(href.rstrip('.,;)'))
    raw = re.findall(r'https://drive\.google\.com/[^\s"<>&]+', html)
    for l in raw:
        c = l.rstrip('.,;)')
        if c not in drive_links:
            drive_links.append(c)
    print(f'\u2705 Brief lu : {len(drive_links)} lien(s) Drive trouves')
    return {'text': text, 'drive_links': drive_links}

brief = read_brief(BRIEF_URL)
print(f'\nApercu du brief :\n{brief["text"][:300]}...')

In [ ]:
# ============================================================
# CELLULE 5 - TELECHARGEMENT DES VIDEOS DEPUIS DRIVE
# ============================================================
# Utilise l API Google Drive (authentification Colab) au lieu de gdown
# pour acceder aux dossiers partages meme sans lien public.
# ============================================================
import time, json
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import google.auth
import io

# Authentification via le compte Google connecte a Colab
auth.authenticate_user()
creds, _ = google.auth.default()
drive_service = build('drive', 'v3', credentials=creds)

VIDEO_EXT = {'.mp4', '.mov', '.avi', '.mkv', '.webm'}

def get_folder_id(url):
    m = re.search(r'/folders/([a-zA-Z0-9_-]+)', url)
    return m.group(1) if m else None

def list_files_in_folder(folder_id):
    files = []
    page_token = None
    while True:
        resp = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            spaces='drive',
            fields='nextPageToken, files(id, name, mimeType, size)',
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True
        ).execute()
        files.extend(resp.get('files', []))
        page_token = resp.get('nextPageToken')
        if not page_token:
            break
    return files

def download_file(file_id, dest_path):
    request = drive_service.files().get_media(fileId=file_id, supportsAllDrives=True)
    with open(dest_path, 'wb') as f:
        downloader = MediaIoBaseDownload(f, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
    return dest_path

seen_folders = set()

for url in brief['drive_links']:
    folder_id = get_folder_id(url)
    if folder_id and folder_id not in seen_folders:
        seen_folders.add(folder_id)
        print(f'\nDossier : {folder_id}')
        try:
            remote_files = list_files_in_folder(folder_id)
            video_count = 0
            for f in remote_files:
                name = f['name']
                ext = '.' + name.rsplit('.', 1)[-1].lower() if '.' in name else ''
                if ext not in VIDEO_EXT:
                    continue
                dest = DOWNLOAD_DIR / name
                if dest.exists() and dest.stat().st_size > 500_000:
                    print(f'  \u2705 {name} (deja telecharge)')
                    video_count += 1
                    continue
                print(f'  Telechargement de {name}...')
                try:
                    download_file(f['id'], str(dest))
                    video_count += 1
                    size_mb = dest.stat().st_size / (1024*1024)
                    print(f'  \u2705 {name} ({size_mb:.1f} MB)')
                except Exception as e:
                    print(f'  \u274c Erreur {name}: {e}')
            print(f'  {video_count} video(s) depuis ce dossier')
        except Exception as e:
            print(f'  \u274c Erreur acces dossier: {e}')

video_files = sorted([
    p for p in DOWNLOAD_DIR.rglob('*')
    if p.suffix.lower() in VIDEO_EXT and p.stat().st_size > 500_000
])
print(f'\n\u2705 {len(video_files)} video(s) prete(s):')
for v in video_files:
    print(f'  {v.name} ({v.stat().st_size/(1024*1024):.1f} MB)')

In [ ]:
# ============================================================
# CELLULE 6 - ANALYSE IA AVEC GEMINI 2.5 FLASH (MODE AUDIO)
# ============================================================
# Mode audio : extrait le son de chaque video et l envoie a Gemini.
# Consomme 50 a 100x moins de tokens que l upload video complete.
# Compatible 100% avec le quota gratuit pour 27+ videos.
# ============================================================
from google import genai
from google.genai import types
import subprocess, json, time, os

client = genai.Client(api_key=GEMINI_API_KEY)

# --- Verification rapide de la cle API ---
try:
    test = client.models.generate_content(
        model='gemini-2.5-flash',
        contents='Reponds uniquement: OK',
        config=types.GenerateContentConfig(temperature=0.0)
    )
    print(f'\u2705 Cle API valide ! (Gemini repond: {test.text.strip()})')
except Exception as e:
    raise RuntimeError(f'\u274c Cle API invalide ou probleme reseau: {e}')

def get_duration(path):
    r = subprocess.run(['ffprobe', '-v', 'quiet', '-print_format', 'json',
                        '-show_format', str(path)], capture_output=True, text=True)
    try: return float(json.loads(r.stdout)['format']['duration'])
    except: return 60.0

def extract_audio(video_path):
    """Extrait l audio en MP3 mono 64kbps (fichier tres leger)."""
    audio_path = str(video_path) + '.mp3'
    if os.path.exists(audio_path) and os.path.getsize(audio_path) > 1000:
        return audio_path
    cmd = [
        'ffmpeg', '-y', '-i', str(video_path),
        '-vn',  # pas de video
        '-ac', '1',  # mono
        '-ab', '64k',  # 64kbps (tres leger)
        '-ar', '22050',  # 22kHz
        audio_path
    ]
    result = subprocess.run(cmd, capture_output=True, timeout=120)
    if result.returncode != 0:
        raise RuntimeError(f'FFmpeg audio extraction failed')
    size_kb = os.path.getsize(audio_path) / 1024
    print(f'    Audio extrait: {size_kb:.0f} KB')
    return audio_path

def analyze_video(video_path, n_clips, min_dur, max_dur, brief_text):
    # 1. Extraire l audio
    print(f'  Extraction audio...')
    audio_path = extract_audio(video_path)
    duration = get_duration(video_path)

    # 2. Upload audio (beaucoup plus leger qu une video)
    print(f'  Upload audio vers Gemini...')
    af = client.files.upload(file=audio_path, config={'display_name': video_path.name + '.mp3'})
    while af.state == 'PROCESSING':
        time.sleep(2)
        af = client.files.get(name=af.name)
    if af.state == 'FAILED':
        raise RuntimeError('Gemini ne peut pas traiter cet audio')

    # 3. Prompt guide par le brief
    prompt = f"""Tu es un directeur artistique et monteur video expert specialise dans le contenu viral TikTok et Instagram Reels.

=== BRIEF DE LA CAMPAGNE (DOCUMENT DIRECTEUR) ===
{brief_text}
=== FIN DU BRIEF ===

Tu viens d ecouter l audio d une video de {duration:.0f} secondes.

INSTRUCTIONS :
1. En te basant STRICTEMENT sur le brief ci-dessus, identifie {n_clips} segment(s) audio qui correspondent le mieux aux objectifs, au ton et au message de la campagne.
2. Chaque segment doit durer entre {min_dur} et {max_dur} secondes.
3. Privilegier les moments avec : parole engageante, emotion, punchlines, musique forte, ou contenu en lien direct avec le brief.
4. Pour chaque segment, genere :
   - Un caption (max 10 mots) : texte d accroche percutant et viral, fidele au brief.
   - Une description : texte de post TikTok/Reels engageant avec des hashtags pertinents au brief.

REPONDS UNIQUEMENT en JSON valide, sans texte autour :
{{{{
  "clips": [
    {{{{
      "id": 1,
      "start": 10.0,
      "end": 25.0,
      "duration": 15.0,
      "score": 95,
      "reason": "Pourquoi ce moment correspond au brief",
      "caption": "Texte viral court",
      "description": "Description engageante pour le post avec #hashtags conformes au brief"
    }}}}
  ]
}}}}"""

    # 4. Appel API avec retry intelligent
    max_retries = 5
    for attempt in range(max_retries):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=[af, prompt],
                config=types.GenerateContentConfig(
                    temperature=0.2,
                    response_mime_type='application/json'
                )
            )
            raw = resp.text.strip()
            break
        except Exception as e:
            err_str = str(e)
            if '429' in err_str or '503' in err_str:
                wait = 60 * (attempt + 1)
                import re as _re
                delay_match = _re.search(r'retryDelay.*?(\\d+)s', err_str)
                if delay_match:
                    wait = int(delay_match.group(1)) + 5
                print(f'    \u23f3 Quota API. Attente {wait}s (tentative {attempt+1}/{max_retries})...')
                time.sleep(wait)
            else:
                try: client.files.delete(name=af.name)
                except: pass
                raise RuntimeError(f'Erreur API: {err_str[:300]}')
    else:
        try: client.files.delete(name=af.name)
        except: pass
        raise RuntimeError('Quota API epuise apres 5 tentatives. Reessayez dans 1h.')

    # 5. Nettoyage
    try: client.files.delete(name=af.name)
    except: pass

    # 6. Parse JSON
    raw = raw.replace('```json', '').replace('```', '').strip()
    try:
        data = json.loads(raw)
        clips = data.get('clips', [])
        if not clips:
            raise RuntimeError(f'Reponse vide de Gemini: {raw[:200]}')
        return clips
    except json.JSONDecodeError:
        raise RuntimeError(f'JSON invalide: {raw[:200]}')

# --- Boucle principale ---
durations = {str(v): get_duration(v) for v in video_files}
total_dur = sum(durations.values())
analysis_results = {}
brief_text_for_ai = brief['text'][:3000]

for i, video_path in enumerate(video_files):
    if i > 0:
        print(f'  \u23f3 Pause 8s (respect quota gratuit)...')
        time.sleep(8)
    dur = durations[str(video_path)]
    n = max(1, round((dur / total_dur) * CLIPS_TO_GENERATE)) if total_dur > 0 else 1
    n = min(n, max(1, int(dur // MIN_CLIP_DURATION)))
    print(f'\n[{i+1}/{len(video_files)}] {video_path.name} -> {n} clips ({dur:.0f}s)')
    try:
        clips = analyze_video(video_path, n, MIN_CLIP_DURATION, MAX_CLIP_DURATION, brief_text_for_ai)
        analysis_results[str(video_path)] = {'clips': clips, 'error': None}
        print(f'  \u2705 {len(clips)} clips trouves')
    except Exception as e:
        print(f'  \u274c ERREUR: {e}')
        analysis_results[str(video_path)] = {'clips': [], 'error': str(e)}

ok_count = sum(1 for v in analysis_results.values() if v['clips'])
err_count = sum(1 for v in analysis_results.values() if v['error'])
total = sum(len(v['clips']) for v in analysis_results.values())
print(f'\n{"="*50}')
print(f'  \u2705 {ok_count} videos analysees avec succes')
if err_count: print(f'  \u274c {err_count} videos en erreur')
print(f'  \ud83c\udfac {total} clips identifies par Gemini')
print(f'{"="*50}')


In [ ]:
# ============================================================
# CELLULE 7 - MONTAGE VIDEO FFMPEG (9:16 vertical)
# ============================================================
from datetime import datetime

campaign_name = 'soul_tied'
created_clips = []

for video_path_str, data in analysis_results.items():
    clips = data.get('clips', [])
    if not clips:
        continue
    video_path = Path(video_path_str)
    dur_total  = durations[video_path_str]
    for clip in clips:
        start    = max(0.0, float(clip.get('start', 0)))
        end      = min(float(clip.get('end', start+30)), dur_total)
        if end <= start:
            print(f'  \u26a0\ufe0f Clip ignore (start={start}, end={end})')
            continue
        cid      = clip.get('id', len(created_clips)+1)
        ts       = datetime.now().strftime('%H%M%S')
        out_name = f'{campaign_name}_clip_{cid:02d}_{ts}.mp4'
        out_path = OUTPUT_DIR / out_name
        cmd = [
            'ffmpeg', '-y', '-ss', str(start), '-i', str(video_path),
            '-t', str(end - start),
            '-vf', 'scale=-2:1920,crop=1080:1920',
            '-c:v', 'libx264', '-preset', 'fast', '-crf', '23',
            '-c:a', 'aac', '-b:a', '128k', '-movflags', '+faststart',
            str(out_path)
        ]
        result = subprocess.run(cmd, capture_output=True, timeout=300)
        if result.returncode == 0 and out_path.exists():
            size_mb = out_path.stat().st_size / (1024*1024)
            print(f'  \u2705 {out_name} ({size_mb:.1f} MB)')

            # Sauvegarder la caption et la description
            txt_path = out_path.with_suffix('.txt')
            with open(txt_path, 'w', encoding='utf-8') as txt_file:
                txt_file.write(f'--- CAPTION (Texte sur la video) ---\n')
                txt_file.write(f'{clip.get("caption", "")}\n\n')
                txt_file.write(f'--- DESCRIPTION (Pour le post) ---\n')
                txt_file.write(f'{clip.get("description", "")}\n')

            created_clips.append({
                'path': out_path,
                'caption': clip.get('caption', ''),
                'description': clip.get('description', ''),
                'score': clip.get('score', '?'),
                'reason': clip.get('reason', ''),
                'source': video_path.name
            })
        else:
            err_msg = result.stderr.decode('utf-8', errors='replace')[-200:] if result.stderr else 'inconnu'
            print(f'  \u274c ERREUR ffmpeg clip {cid}: {err_msg}')

print(f'\n\u2705 {len(created_clips)} clips crees dans Google Drive !')

In [ ]:
# ============================================================
# CELLULE 8 - RAPPORT FINAL + ENVOI SUR GITHUB
# ============================================================
import base64
from datetime import datetime

now = datetime.now().strftime('%Y-%m-%d %H:%M')

# Construire le rapport
lines = [
    f'# Rapport - AI Clipping Bot',
    f'**Date :** {now}',
    f'**Brief :** {BRIEF_URL}',
    '',
    '## R\u00e9sultats',
    f'- Videos analysees : **{len(video_files)}**',
    f'- Clips generes    : **{len(created_clips)}**',
    f'- Dossier Drive    : `{DRIVE_OUTPUT_FOLDER}`',
    '',
    '## Clips cr\u00e9\u00e9s',
]
for c in created_clips:
    p = c['path']
    mb = p.stat().st_size / (1024*1024)
    lines.append(f'### `{p.name}` ({mb:.1f} MB)')
    lines.append(f'- **Source :** {c["source"]}')
    lines.append(f'- **Score :** {c["score"]}')
    lines.append(f'- **Raison :** {c["reason"]}')
    lines.append(f'- **Caption :** {c["caption"]}')
    lines.append(f'- **Description :** {c["description"]}')
    lines.append('')

lines += [
    '',
    '## Analyse par video',
]
for vp_str, data in analysis_results.items():
    vname = Path(vp_str).name
    clips = data['clips']
    err = data['error']
    if err:
        lines.append(f'\n### \u274c {vname} (ERREUR)')
        lines.append(f'> {err[:200]}')
    else:
        lines.append(f'\n### \u2705 {vname} ({len(clips)} clips)')
        for c in clips:
            lines.append(f'- Clip {c.get("id")}: {c.get("start",0):.1f}s -> {c.get("end",0):.1f}s | Score: {c.get("score","?")} | {c.get("reason","")}')

report_md = '\n'.join(lines)
print(report_md[:800])

# Envoyer sur GitHub
gh_headers = {
    'Authorization': f'token {GITHUB_TOKEN}',
    'Accept': 'application/vnd.github.v3+json'
}
report_filename = f'rapports/rapport_{datetime.now().strftime("%Y%m%d_%H%M%S")}.md'
content_b64 = base64.b64encode(report_md.encode()).decode()

r = requests.put(
    f'https://api.github.com/repos/{GITHUB_USER}/{GITHUB_REPO}/contents/{report_filename}',
    headers=gh_headers,
    json={'message': f'Rapport {now}', 'content': content_b64}
)

if r.status_code in (200, 201):
    print(f'\n\u2705 Rapport envoye sur GitHub !')
    print(f'   https://github.com/{GITHUB_USER}/{GITHUB_REPO}/tree/main/rapports')
else:
    print(f'\n\u26a0\ufe0f GitHub non disponible ({r.status_code})')

print('\n' + '='*50)
print(f'  TERMINE ! {len(created_clips)} clips dans votre Drive.')
print('='*50)